In [1]:
from qwen_vl_utils import process_vision_info
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor

/ubc/cs/research/beaver/projects/carlos/conda_envs/vlmcot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load model
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)

# Load the processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

Loading weights: 100%|██████████| 729/729 [00:02<00:00, 287.99it/s]


## HF card tutorial

In [3]:
# Try with the example from the HF card
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {
                "type": "text",
                "text": "Describe this image. At the end of your message say 'hello'. "
            },
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

In [4]:
# Generate output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(output_text)

['In the image, a person and a dog are sitting on a sandy beach during what appears to be either sunrise or sunset, as indicated by the warm, golden light. The person is wearing a plaid shirt and dark pants, and they are smiling while interacting with the dog. The dog, which seems to be a Labrador Retriever, is wearing a colorful harness and is giving a high-five to the person. The ocean is visible in the background, with gentle waves rolling onto the shore. The overall atmosphere of the image is serene and joyful. hello']


In [5]:
# Let's functionalize this
# TODO: for token_len, we have 128 but for COT we will need more? How do we decide this?
def generate_output(image_path, text_prompt, token_len=128):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                },
                {
                    "type": "text",
                    "text": text_prompt
                },
            ],
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to("cuda")

    # Greedy decoding (always the most likely token) -> same output every run.
    # Overrides Qwen's defaults (sampling + repetition_penalty=1.05).
    generated_ids = model.generate(**inputs, max_new_tokens=token_len, do_sample=False, repetition_penalty=1.0)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return output_text[0]

## Test CoT with datasets

In [6]:
import sys
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
sys.path.append(str(DATA_DIR))
from data import load_processed

sqa_data = load_processed("scienceqa")
aok = load_processed("aokvqa")
data = sqa_data + aok

In [7]:
# Prep the prompts: we have a Q, and give it options to choose from
OPTN_LETTERS = "ABCDE"

# Direct or CoT
DIRECT = "\nAnswer with the option's letter only."
COT = "\nThink step by step. Write each reasoning step on its own numbered line. Then write the final line as 'Answer: <letter>'."

for ex in data:
    prompt = ex["question"]
    # TODO: do we want to add hint?
    if ex["context"] != "": 
        prompt += f"\nHint: {ex['context']}"
    
    choices = "\nOptions: "
    for i, c in enumerate(ex["choices"]):
        choices += f"{OPTN_LETTERS[i]}) {c} "

    prompt += choices

    # TODO thoughts on this?
    ex["direct_prompt"] = prompt + DIRECT
    ex["COT_prompt"] = prompt + COT

In [9]:
# Now ask the question, get the CoT, for a few Qs
for ex in data[:3]:
    ex_out = generate_output(
        image_path=ex["image_path"], 
        text_prompt=ex["COT_prompt"],
        token_len=1024
    )
    print(f"Q: {ex['COT_prompt']}\n\nA: {ex_out}\n=====\n")

Q: What is the capital of Florida?
Options: A) Orlando B) Baton Rouge C) Tallahassee D) Tampa 
Think step by step. Write each reasoning step on its own numbered line. Then write the final line as 'Answer: <letter>'.

A: 1. The map shows the United States, with Florida highlighted in a darker green.
2. The question asks for the capital of Florida.
3. Among the given options, A) Orlando, B) Baton Rouge, C) Tallahassee, and D) Tampa, we need to identify which one is the capital of Florida.
4. Orlando is a major city in Florida but not the capital.
5. Baton Rouge is the capital of Louisiana, not Florida.
6. Tampa is a major city in Florida but not the capital.
7. Tallahassee is the capital of Florida.

Answer: C
=====

Q: Which animal's limbs are also adapted for climbing trees?
Hint: Chimpanzees live in the forests of Central Africa. Their limbs are adapted for climbing trees.
Figure: chimpanzee.
Options: A) three-toed sloth B) sea turtle 
Think step by step. Write each reasoning step on 

## Save results for the 20 examples

In [10]:
import json
import re

# Remove the answer from the rationale
def parse_answer(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text) or re.match(r"\s*\(?([A-E])\b", text)
    return m.group(1) if m else None

# Separate each numbered rationale so we can manipulate more easily
def split_steps(text):
    body = text.split("Answer:")[0]
    return [s.strip() for s in re.findall(r"^\s*\d+\.\s*(.+)$", body, flags=re.M)]

# Run CoT on all examples and save one record per question ID
RESULTS_DIR = Path.cwd().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

with open(RESULTS_DIR / "cot_baseline.jsonl", "w") as f:
    for ex in data:
        # generate output w/ CoT prompt
        ex_out = generate_output(
            image_path=ex["image_path"],
            text_prompt=ex["COT_prompt"],
            token_len=1024
        )

        # parse answers: save steps, save true answer, save pred answer
        pred = parse_answer(ex_out)
        gold = OPTN_LETTERS[ex["answer_idx"]]
        steps = split_steps(ex_out)
        f.write(json.dumps({
            "id": ex["id"],
            "dataset": ex["dataset"],
            "prompt": ex["COT_prompt"],
            "output": ex_out,
            "steps": steps,
            "pred": pred,
            "gold": gold,
            "correct": pred == gold,
        }) + "\n")
        print(ex["id"], "pred:", pred, "gold:", gold, "steps:", len(steps))

sqa_test_2076 pred: C gold: C steps: 7
sqa_test_150 pred: A gold: A steps: 6
sqa_test_3450 pred: A gold: A steps: 4
sqa_test_1650 pred: C gold: C steps: 7
sqa_test_3665 pred: A gold: A steps: 5
sqa_test_2129 pred: B gold: C steps: 6
sqa_test_3243 pred: B gold: B steps: 5
sqa_test_2531 pred: A gold: A steps: 4
sqa_test_681 pred: D gold: D steps: 4
sqa_test_1666 pred: C gold: C steps: 3
aok_QsW8sBQzRc4Zsxds7WwTJ7 pred: D gold: D steps: 5
aok_FG6kmHCaikmwGGpw9RhpSD pred: C gold: C steps: 4
aok_PCavtGPpEZnoW3vRUb2vbM pred: A gold: A steps: 5
aok_nqDVieEVdjDt3CELi7g5Zh pred: C gold: C steps: 5
aok_n3q6CBsJfobMm4vAqMGi38 pred: A gold: A steps: 4
aok_n3EWd223KWajDuTgFCGZmw pred: C gold: C steps: 4
aok_eBgzKWWRWbkzRJsrfBqGBV pred: A gold: A steps: 4
aok_5jGPL4u8JFHkLyQCG354uG pred: B gold: B steps: 4
aok_FFhuFfMAy38Z7WWr6q7faA pred: D gold: A steps: 5
aok_99Ra8z27K9FLU2u5acKAqp pred: A gold: A steps: 3


In [11]:
# Same for the direct prompt (answer only, no rationale) -> results/direct.jsonl
with open(RESULTS_DIR / "direct.jsonl", "w") as f:
    for ex in data:
        ex_out = generate_output(
            image_path=ex["image_path"],
            text_prompt=ex["direct_prompt"],
            token_len=128 # less tokens than w/ CoT 128 is defauly i think
        )
        pred = parse_answer(ex_out)
        gold = OPTN_LETTERS[ex["answer_idx"]]
        f.write(json.dumps({
            "id": ex["id"],
            "dataset": ex["dataset"],
            "prompt": ex["direct_prompt"],
            "output": ex_out,
            "pred": pred,
            "gold": gold,
            "correct": pred == gold,
        }) + "\n")
        print(ex["id"], "output:", repr(ex_out), "pred:", pred, "gold:", gold)

sqa_test_2076 output: 'C' pred: C gold: C
sqa_test_150 output: 'A' pred: A gold: A
sqa_test_3450 output: 'A' pred: A gold: A
sqa_test_1650 output: 'C' pred: C gold: C
sqa_test_3665 output: 'A' pred: A gold: A
sqa_test_2129 output: 'A' pred: A gold: C
sqa_test_3243 output: 'B)' pred: B gold: B
sqa_test_2531 output: 'A' pred: A gold: A
sqa_test_681 output: 'D' pred: D gold: D
sqa_test_1666 output: 'C' pred: C gold: C
aok_QsW8sBQzRc4Zsxds7WwTJ7 output: 'D' pred: D gold: D
aok_FG6kmHCaikmwGGpw9RhpSD output: 'C' pred: C gold: C
aok_PCavtGPpEZnoW3vRUb2vbM output: 'A' pred: A gold: A
aok_nqDVieEVdjDt3CELi7g5Zh output: 'C' pred: C gold: C
aok_n3q6CBsJfobMm4vAqMGi38 output: 'A' pred: A gold: A
aok_n3EWd223KWajDuTgFCGZmw output: 'C' pred: C gold: C
aok_eBgzKWWRWbkzRJsrfBqGBV output: 'A' pred: A gold: A
aok_5jGPL4u8JFHkLyQCG354uG output: 'B' pred: B gold: B
aok_FFhuFfMAy38Z7WWr6q7faA output: 'A' pred: A gold: A
aok_99Ra8z27K9FLU2u5acKAqp output: 'A' pred: A gold: A


## Start `answer_given_rationale`